In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,START,END
from pydantic import BaseModel,Field
from typing import Literal

In [9]:
### CODING,GOOGLE_SEARCH,WEATHER
class FlowState(BaseModel):
    question:str=Field(description="User Asked Question")
    category:Literal['coding','google_search','weather']=Field(default="google_search")
    answer:str=Field(default="")

In [10]:
class QuestionCategory(BaseModel):
    category:Literal['coding','google_search','weather']=Field(default="google_search",
                                                               description="Question Category")

In [ ]:
llm=ChatGroq(model="llama-3.3-70b-versatile")

In [ ]:
def check_question_category(state:FlowState)->FlowState:
    st_llm=llm.with_structured_output(QuestionCategory)
    res=st_llm.invoke(f"I want to know the category of my question,question is:{state.question}.If you are not sure then just give 'google_search' as a category")
    print(res)
    state.category=res.category
    
    return state

In [ ]:
def route(state:FlowState)-> Literal['coding','google_search','weather']:
    return state.category

In [ ]:
def coding_node(state:FlowState)->FlowState:
    res=llm.invoke("you are a coding expert:{state.question}")
    state.answer=res.content
    return state


def weather_node(state:FlowState)->FlowState:
    res=llm.invoke("you are weather expert:{state.question}")
    state.answer=res.content
    return state


def google_search_node(state:FlowState)->FlowState:
    res=llm.invoke("you are google search expert:{state.question}")
    state.answer=res.content
    return state

In [ ]:
graph=StateGraph(FlowState)

graph.add_node("check_question_category",check_question_category)
graph.add_node("coding", coding_node)
graph.add_node("weather", weather_node)
graph.add_node("google_search", google_search_node)

graph.add_edge(START,"check_question_category")
graph.add_conditional_edge("check_question_category",route)
graph.add_edge("coding", END)
graph.add_edge("weather", END)
graph.add_edge("google_search", END)

graph=graph.compile()


In [ ]:
from IPython.display import Image
Image(graph.get_graph().draw_mermaid_png())

In [ ]:
response=graph.invoke({"question":"What is the weather now"})